# Dataset preparation

In [8]:
cd ..

 all-train.out                         multimodal-model-balanced.pt
 all-train2.out                        multimodal-model.pt
 analysis.out                          notebooks/
 best-models/                          old-models/
 best_trial.pt                         ollama.out
 bimodal-train.out                     pipeline/
 column_i2b2/                          precompute.sbatch
 column_thyme/                         precompute_graphs_for_analysis.sbatch
 computed_kg.pt                        pregenerated/
'construct graph from text only.log'  'pretrained models'/
 custom_datasets/                      resources/
 data/                                 results/
 dataLoaders/                          results-baselineBERT/
 dataset_loaders/                      results-bimodal/
 dataset_train.pt                      results-glm/
 date2vec/                             results-graph/
 demofile2.txt                         results-text/
 demofile3.txt                         results_text/


In [2]:
!pip install -q seqeval
!pip install -q wandb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.1 requires accelerate==0.33.0, but you have accelerate 0.11.0 which is incompatible.
pytdc 1.1.1 requires transformers==4.43.4, but you have transformers 4.20.1 which is incompatible.
lightning 2.0.8 requires pydantic<2.2.0,>=1.7.4, but you have pydantic 2.10.3 which is incompatible.


In [9]:
from training.train_event_extraction import *

ModuleNotFoundError: No module named 'training.train_event_extraction'

In [4]:
def load_stored_dataset_combination_graph(balanced=True, dataset="i2b2"):
    dataset_train = DFDataset()
    dataset_train.load("pregenerated/"+dataset+"_dataset_train_rawkg.pt")
    dataset_val = DFDataset()
    dataset_val.load("pregenerated/"+dataset+"_dataset_val_rawkg.pt")
    dataset_test = DFDataset()
    dataset_test.load("pregenerated/"+dataset+"_dataset_test_rawkg.pt")

    if balanced:
        dataset_train.oversample_pregenerated()
        dataset_val.oversample_pregenerated()

    return dataset_train, dataset_val, dataset_test

In [5]:
# test_name = "thyme"
test_name = "i2b2"
dataset = load_stored_dataset_combination_graph(balanced=False, dataset=test_name)
dataset_train, dataset_val, dataset_test = dataset

NameError: name 'DFDataset' is not defined

In [6]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [7]:
def sentence_spans(txt):
    tokens=nltk.sent_tokenize(txt)
    offset = 0
    for token in tokens:
        offset = txt.find(token, offset)
        yield token, offset, offset+len(token)
        offset += len(token)

In [8]:
def word_spans(txt, start_offset=0):
    tokens=nltk.word_tokenize(txt)
    offset = 0
    for token in tokens:
        offset = txt.find(token, offset)
        yield token, offset+start_offset, offset+len(token)+start_offset
        offset += len(token)

In [9]:
def event_tag(token_span, events):
    for event in events:
        event_text, event_start, event_end = event
        token_text, token_start, token_end = token_span
        if event_start < token_end and event_start >= token_start:
            return "B"
        elif token_start >= event_start and token_start < event_end:
            return "I"
    return "X"

In [10]:
def convert_dataframe_to_column_format(df, filename):
    f = open(filename, "w")
    documents = set(df["document_id"])
    for doc in documents:
        relevant_rows = df[df["document_id"] == doc]
        text = relevant_rows.iloc[0]["text"]
        events = set()
        for row in relevant_rows.iloc:
            events.add((row["event1_text"], row["event1_start"], row["event1_end"]))
            events.add((row["event2_text"], row["event2_start"], row["event2_end"]))
        # print(text, events)
        
        for sentence_span in sentence_spans(text):
            words = list(word_spans(sentence_span[0], sentence_span[1]))
            tagged = nltk.pos_tag([span[0] for span in words])
            for i, word_span in enumerate(words):
                if len(word_span[0]) > 0:
                    # print(word_span[0], tagged[i][1], event_tag(word_span, events))
                    f.write(word_span[0].replace(" ", "_").replace("''",'"') +"\t"+ event_tag(word_span, events) + "\n")
            # print()
            f.write("\n")
    f.close()
# import os
# os.mkdir("column_i2b2")
convert_dataframe_to_column_format(dataset_train.df, "column_"+test_name+"/train.txt")
convert_dataframe_to_column_format(dataset_val.df, "column_"+test_name+"/val.txt")
convert_dataframe_to_column_format(dataset_test.df, "column_"+test_name+"/test.txt")

In [11]:
!pip install flair==0.12.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.5/788.5 kB 21.9 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 82.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... one
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.1/373.1 kB 10.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 90.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 20.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 102.6 MB/s eta 0:

In [12]:
from flair.data import Corpus
from flair.datasets import ColumnCorpus

# define columns
#columns = {0: 'text', 1: 'pos', 2: 'ner'}
columns = {0: 'text', 1: 'ner'}

# this is the folder in which train, test and dev files reside
data_folder = 'column_' + test_name

# init a corpus using column format, data folder and the names of the train, dev and test files
corpus: Corpus = ColumnCorpus(data_folder, columns,
                              train_file='train.txt',
                              test_file='test.txt',
                              dev_file='val.txt')


2024-12-10 13:14:59,868 Reading data from column_i2b2
2024-12-10 13:14:59,869 Train: column_i2b2/train.txt
2024-12-10 13:14:59,870 Dev: column_i2b2/val.txt
2024-12-10 13:14:59,870 Test: column_i2b2/test.txt


In [15]:
for i in range(20):
    print(corpus.train[0][i])

Token[0]: "ADMISSION" → B (1.0)
Token[1]: "DATE" → X (1.0)
Token[2]: ":" → X (1.0)
Token[3]: "5-30-93" → X (1.0)
Token[4]: "DISCHARGE" → B (1.0)
Token[5]: "DATE" → X (1.0)
Token[6]: ":" → X (1.0)
Token[7]: "6-2-93" → X (1.0)
Token[8]: "DISCHARGE" → X (1.0)
Token[9]: "DATE" → X (1.0)
Token[10]: ":" → X (1.0)
Token[11]: "6-2-93" → X (1.0)
Token[12]: "HISTORY" → X (1.0)
Token[13]: "OF" → X (1.0)
Token[14]: "PRESENT" → X (1.0)
Token[15]: "ILLNESS" → X (1.0)
Token[16]: ":" → X (1.0)
Token[17]: "The" → X (1.0)
Token[18]: "patient" → X (1.0)
Token[19]: "is" → X (1.0)


# Train the model

In [16]:
from flair.datasets import CONLL_03
from flair.embeddings import TransformerWordEmbeddings
from flair.models import SequenceTagger
from flair.trainers import ModelTrainer

# 1. get the corpus
print(corpus)

# 2. what label do we want to predict?
label_type = 'ner'

# 3. make the label dictionary from the corpus
label_dict = corpus.make_label_dictionary(label_type=label_type, add_unk=False)
print(label_dict)

# 4. initialize fine-tuneable transformer embeddings WITH document context
embeddings = TransformerWordEmbeddings(model='xlm-roberta-large',
                                       layers="-1",
                                       subtoken_pooling="first",
                                       fine_tune=True,
                                       use_context=True,
                                       )

# 5. initialize bare-bones sequence tagger (no CRF, no RNN, no reprojection)
tagger = SequenceTagger(hidden_size=256,
                        embeddings=embeddings,
                        tag_dictionary=label_dict,
                        tag_type='ner',
                        use_crf=False,
                        use_rnn=False,
                        reproject_embeddings=False,
                        )

# 6. initialize trainer
trainer = ModelTrainer(tagger, corpus)

# 7. run fine-tuning
trainer.fine_tune('resources/taggers/sota-ner-flert-'+test_name + '2',
                  learning_rate=5.0e-6,
                  mini_batch_size=4,
                  # mini_batch_chunk_size=1,  # remove this parameter to speed up computation if you have a big GPU
                  )

Corpus: 4099 train + 1278 dev + 4602 test sentences
2024-12-10 13:21:42,563 Computing label dictionary. Progress:


4099it [00:00, 32380.18it/s]

2024-12-10 13:21:42,695 Dictionary created for label 'ner' with 3 values: X (seen 45169 times), I (seen 12189 times), B (seen 10699 times)
Dictionary with 3 tags: X, I, B


Downloading:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/616 [00:00<?, ?B/s]

...

2024-12-10 13:55:17,839 Evaluating as a multi-label problem: False


2024-12-10 13:55:17,925 DEV : loss 0.5344463586807251 - f1-score (micro avg)  0.8905
2024-12-10 13:55:23,902 ----------------------------------------------------------------------------------------------------
2024-12-10 13:55:23,905 Testing using last state of model ...



00%|██████████████████████████████████████████████████████████████████████| 1151/1151 [01:08<00:00, 16.85it/s]

2024-12-10 13:56:32,358 Evaluating as a multi-label problem: False
2024-12-10 13:56:32,671 0.9027	0.9027	0.9027	0.9027
2024-12-10 13:56:32,672 
Results:
- F-score (micro) 0.9027
- F-score (macro) 0.8726
- Accuracy 0.9027

By class:
              precision    recall  f1-score   support

           X     0.9485    0.9181    0.9330     54925
           I     0.8246    0.8651    0.8444     13839
           B     0.8068    0.8769    0.8404     12464

    accuracy                         0.9027     81228
   macro avg     0.8600    0.8867    0.8726     81228
weighted avg     0.9057    0.9027    0.9037     81228

2024-12-10 13:56:32,672 ----------------------------------------------------------------------------------------------------


{'test_score': 0.9027305855123849,
 'dev_score_history': [0.8731965890758239,
  0.8873934086194976,
  0.8931090112929246,
  0.8938926019820235,
  0.8938465084120765,
  0.8917262041945149,
  0.8927863563032957,
  0.8938926019820235,
  0.8911269877852039,
  0.8904816778059461],
 'train_loss_history': [0.7396776843198326,
  0.3405366600604619,
  0.2720411952524168,
  0.23437428666621624,
  0.2023811094865851,
  0.17654571932443106,
  0.15770144470607822,
  0.13887975254687027,
  0.1267616977996426,
  0.12291113930872227],
 'dev_loss_history': [0.39632776379585266,
  0.360276997089386,
  0.3520735204219818,
  0.36324363946914673,
  0.4016031324863434,
  0.3765728175640106,
  0.4165569543838501,
  0.49318593740463257,
  0.49572041630744934,
  0.5344463586807251]}

# Predicting

In [11]:
from flair.embeddings import TransformerWordEmbeddings
from flair.models import SequenceTagger
from flair.trainers import ModelTrainer
from flair.data import Sentence

# load the model you trained
# model = SequenceTagger.load('resources/taggers/sota-ner-flert/final-model.pt')
model = SequenceTagger.load('resources/taggers/sota-ner-flert-'+test_name+'/final-model.pt')

# create example sentence
sentence = Sentence('I love Berlin')

# predict tags and print
model.predict(sentence)

print(sentence.to_tagged_string())

2024-12-11 09:40:32,847 SequenceTagger predicts: Dictionary with 3 tags: X, I, B
Sentence[3]: "I love Berlin" → ["I"/X, "love"/X, "Berlin"/X]


In [19]:
sentence[0]

Token[0]: "I" → X (0.9993)

In [20]:
def get_predictions(tagged_sentence, original_text):
    predicted_events = []
    curent_event = [0,0]
    in_event = False
    offset = 0
    for word in tagged_sentence:
        offset = original_text.find(word.text, offset)
        if word.tag == "B":
            if in_event:
                # complete curent event
                predicted_events.append((curent_event[0], curent_event[1], original_text[curent_event[0]: curent_event[1]]))
                current_event = [offset, offset]
                in_event = False
            in_event = True
            curent_event[0] = offset
            curent_event[1] = offset + len(word.text)
        elif word.tag == "I" and in_event == True:
            curent_event[1] = offset + len(word.text)
        elif word.tag == "I" and in_event == False:
            # ignore events without a beginning
            # in_event = True
            # curent_event[0] = offset
            # curent_event[1] = offset + len(word.text)
            pass
        elif word.tag == "X" and in_event == True:
            # complete curent event
            predicted_events.append((curent_event[0], curent_event[1], original_text[curent_event[0]: curent_event[1]]))
            curent_event = [offset, offset]
            in_event = False
        
        
        offset += len(word.text)
    return predicted_events

In [21]:
dataframe = dataset_test.df
documents = set(dataframe["document_id"])
dataset = []
for doc in documents:
    relations = dataframe[dataframe["document_id"] == doc]
    text = relations["text"].iloc[0]
    graph = []
    events = set()
    for row in relations.iloc:
        events.add((row["event1_start"], row["event1_end"], row["event1_text"]))
        events.add((row["event2_start"], row["event2_end"], row["event2_text"]))
        graph.append((row["event1_text"], row["class"], row["event2_text"]))
    dataset.append({"text": text, "graph": graph, "events": events})

In [22]:
print(dataset[0]["text"])


ADMISSION DATE :
12/16/97
DISCHARGE DATE :
01/16/98
HISTORY OF PRESENT ILLNESS :
The patient was a 53-year-old male with a longstanding history of renal disease with multiple complications and problems related to vascular access and hypercoagulable state .
These culminated in an attempted cadaveric kidney transplant undertaken on 12/16/97 , which was complicated by thrombosis and necrosis of the cadaveric renal vein within 24 hours .
This required a transplant nephrectomy .
HOSPITAL COURSE AND TREATMENT :
The patient napos;s course was subsequently characterized by recurrent pneumonias , possibly due to aspiration , and by hemodynamic instability .
He was transferred to the Medical Intensive Care Unit from the Transplant Service for ventilatory management in the setting of hypotension .
On transfer , the patient was not sedated , but was unresponsive and extremely dyssynchronous with the ventilator .
The patient was found to have substantial auto PEEP with an elevated dead space .
A p

In [23]:
def evaluate(ground_truth, predictions):
    def overlaps(event1, event2):
        if event1[0]>= event2[0] and event1[0]<=event2[1]:
            return True
        if event1[1]>= event2[0] and event1[1]<=event2[1]:
            return True
        if event2[0]>= event1[0] and event2[0]<=event1[1]:
            return True
        return False

    false_negatives = [True for _ in range(len(ground_truth))]
    true_positives = 0
    false_positives = 0
    for prediction in predictions:
        overlap = False
        for i, gt in enumerate(ground_truth):
            if overlaps(gt, prediction):
                overlap = True
                false_negatives[i] = False
        if overlap:
            true_positives += 1
        else:
            false_positives += 1
    false_negatives = sum(false_negatives)

    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / len(ground_truth)
    fscore = 2*(precision * recall)/(precision + recall)
    return true_positives, false_positives, len(ground_truth), precision, recall, fscore

In [24]:
text = dataset[0]["text"]
sentence = Sentence(text)
model.predict(sentence)
predictions = get_predictions(sentence, text)
predictions

[(1, 10, 'ADMISSION'),
 (27, 36, 'DISCHARGE'),
 (148, 161, 'renal disease'),
 (167, 189, 'multiple complications'),
 (194, 202, 'problems'),
 (234, 255, 'hypercoagulable state'),
 (278, 318, 'an attempted cadaveric kidney transplant'),
 (384, 420, 'necrosis of the cadaveric renal vein'),
 (453, 477, 'a transplant nephrectomy'),
 (512, 538, 'The patient napos;s course'),
 (573, 593, 'recurrent pneumonias'),
 (612, 622, 'aspiration'),
 (632, 655, 'hemodynamic instability'),
 (665, 676, 'transferred'),
 (680, 711, 'the Medical Intensive Care Unit'),
 (717, 739, 'the Transplant Service'),
 (744, 766, 'ventilatory management'),
 (785, 796, 'hypotension'),
 (802, 810, 'transfer'),
 (833, 840, 'sedated'),
 (851, 863, 'unresponsive'),
 (868, 892, 'extremely dyssynchronous'),
 (898, 912, 'the ventilator'),
 (931, 936, 'found'),
 (945, 966, 'substantial auto PEEP'),
 (972, 994, 'an elevated dead space'),
 (997, 1020, 'A pulmonary arteriogram'),
 (1033, 1045, 'demonstrated'),
 (1046, 1071, 'multi

In [25]:
dataset[0]["events"]

{(1, 10, 'ADMISSION'),
 (148, 161, 'renal disease'),
 (214, 229, 'vascular access'),
 (234, 255, 'hypercoagulable state'),
 (278, 318, 'an attempted cadaveric kidney transplant'),
 (512, 538, 'The patient napos;s course'),
 (573, 593, 'recurrent pneumonias'),
 (612, 622, 'aspiration'),
 (632, 655, 'hemodynamic instability'),
 (665, 676, 'transferred'),
 (680, 711, 'the Medical Intensive Care Unit'),
 (717, 739, 'the Transplant Service'),
 (744, 766, 'ventilatory management'),
 (785, 796, 'hypotension'),
 (802, 810, 'transfer'),
 (833, 840, 'sedated'),
 (851, 863, 'unresponsive'),
 (868, 892, 'extremely dyssynchronous'),
 (898, 912, 'the ventilator'),
 (945, 966, 'substantial auto PEEP'),
 (972, 994, 'an elevated dead space'),
 (997, 1020, 'A pulmonary arteriogram'),
 (1033, 1045, 'demonstrated'),
 (1046, 1071, 'multiple pulmonary emboli'),
 (1115, 1125, 'introduced'),
 (1141, 1166, 'the renal vein thrombosis'),
 (1181, 1209, 'An inferior vena cava filter'),
 (1214, 1220, 'placed'),
 (1

In [26]:
evaluate(dataset[0]["events"], predictions)

(99, 13, 120, 0.8839285714285714, 0.825, 0.8534482758620688)

In [28]:
model = SequenceTagger.load('resources/taggers/sota-ner-flert-'+test_name+'2/final-model.pt')

all_true_positives = 0
all_false_positives = 0
all_gt_positives = 0
for example in dataset:
    text = example["text"]
    sentence = Sentence(text)
    # predict tags and print
    model.predict(sentence)
    predictions = get_predictions(sentence, text)
    # print(predictions)
    # print()
    # print(example["events"])
    true_positives, false_positives, gt_positives, precision, recall, fscore = evaluate(example["events"], predictions)
    all_true_positives += true_positives
    all_false_positives += false_positives
    all_gt_positives += gt_positives
# overall scores
precision = all_true_positives / (all_true_positives + all_false_positives)
recall = all_true_positives / all_gt_positives
fscore = 2*(precision * recall)/(precision + recall)
print("precision:", precision)
print("recall:", recall)
print("fscore:", fscore)

2024-12-10 13:59:07,829 SequenceTagger predicts: Dictionary with 3 tags: X, I, B
precision: 0.8494767312402582
recall: 0.9182445442875481
fscore: 0.882523036588657


In [29]:
test_name

'i2b2'